# Controlled provider-backed RAG experiment
Use the shared CLI instead of duplicating strategy implementations. Set credentials in the notebook launch environment or enter them with getpass. Select an existing compatible index; never reuse a BGE index with Gemini embeddings.
The notebook runs real inference only when the execution cell is run. Use a new output filename for each provider/model/configuration.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
from getpass import getpass
root = Path.cwd()
if not (root / 'rag_lab').is_dir():
    root = root.parent
assert (root / 'rag_lab').is_dir(), 'Run from repository root or notebooks/'
provider = 'groq'
model = 'openai/gpt-oss-120b'
if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass('Groq API key: ')
index = root / 'data/indexes/corpus16_pages_v2/dense_bge'
dataset = root / 'eval/heldout_robustness_candidate_30.json'
output = root / 'eval/generated_groq120b_notebook_smoke.jsonl'
assert index.is_dir() and dataset.is_file(), 'Set the actual compatible index and dataset'


In [ ]:
# One question, one strategy; this spends inference allowance.
cmd = [sys.executable, '-m', 'rag_lab.evaluation.generate', '--provider', provider,
       '--model', model, '--strategies', 'crag_v6', '--backends', 'dense',
       '--chroma-path', str(index), '--chroma-collection', 'langchain',
       '--eval-set', str(dataset), '--limit', '1', '--output', str(output)]
subprocess.run(cmd, cwd=root, check=True)
if output.exists():
    rows = [json.loads(line) for line in output.read_text().splitlines() if line.strip()]
    print('Completed rows:', len(rows))
    if rows:
        print(rows[-1]['answer'])
else:
    print('No completed record; inspect the runner output before retrying.')
